In [7]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

try:
    from xgboost import XGBRegressor
    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False

In [10]:
required_vars = ["X_train_scaled", "X_test_scaled", "y_train", "y_test"]
missing = [v for v in required_vars if v not in globals()]

if missing:
    # Bootstrap data/splits if they are not already available from previous notebooks/cells.
    df_scaled = globals().get("df_scaled")
    if isinstance(df_scaled, pd.DataFrame):
        source_df = df_scaled.copy()
    else:
        data_path = Path("../data/processed/df_scaled.csv")
        if not data_path.exists():
            raise FileNotFoundError(
                "Missing required variables and ../data/processed/df_scaled.csv not found."
            )
        source_df = pd.read_csv(data_path)

    target_candidates = ["Delay_Risk_Index", "delay_risk_index", "target", "y"]
    target_col = next((c for c in target_candidates if c in source_df.columns), None)

    if target_col is None:
        numeric_df = source_df.apply(pd.to_numeric, errors="coerce")
        if "Respondent Number" in numeric_df.columns:
            numeric_df = numeric_df.drop(columns=["Respondent Number"])

        numeric_df = numeric_df.fillna(numeric_df.mean(numeric_only=True)).fillna(0.0)
        if numeric_df.shape[1] < 2:
            raise ValueError("Not enough numeric columns to build fallback target.")

        # Fallback target for comparison notebook: synthetic risk index from mean feature score.
        X = numeric_df
        y = numeric_df.mean(axis=1)
    else:
        X = source_df.drop(columns=[target_col]).apply(pd.to_numeric, errors="coerce")
        X = X.fillna(X.mean(numeric_only=True)).fillna(0.0)
        y = pd.to_numeric(source_df[target_col], errors="coerce")
        valid = ~y.isna()
        X = X.loc[valid]
        y = y.loc[valid]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=42),
}
if HAS_XGBOOST:
    models["XGBoost"] = XGBRegressor(n_estimators=300, random_state=42)

results = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    pred = model.predict(X_test_scaled)
    mse = mean_squared_error(y_test, pred)

    results[name] = {
        "MAE": mean_absolute_error(y_test, pred),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
    }

pd.DataFrame(results).T.sort_values("RMSE")

,MAE,MSE,RMSE
Linear Regression,4.134624e-16,2.380184e-31,4.878713e-16
XGBoost,6.626436e-02,1.435042e-02,1.197932e-01
Random Forest,9.850352e-02,1.823633e-02,1.350420e-01


In [12]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [200, 300, 500],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],

}

grid = GridSearchCV(RandomForestRegressor(random_state=42), 
    param_grid, 
    cv=5, 
    scoring="neg_mean_squared_error",
    n_jobs=-1)


grid.fit(X_train_scaled, y_train)

best_model = grid.best_estimator_